# Bài thực hành: Nhận diện khuôn mặt thời gian thực với FaceNet & MTCNN trên Webcam

**Chương 3 - Phần 2**

Notebook này được chia theo từng bước rõ ràng:

1. Chuẩn bị thư viện, đường dẫn và tham số.
2. Truy cập webcam bằng OpenCV.
3. Tích hợp MTCNN để phát hiện khuôn mặt.
4. Dùng FaceNet/InceptionResnetV1 để trích xuất đặc trưng.
5. So sánh cosine similarity theo thời gian thực.
6. Hiển thị `Matched` nếu `similarity > 0.7`, ngược lại hiển thị `Unknown`.

> Lưu ý: webcam thường chạy ổn hơn khi chạy bằng script `src/realtime_facenet_mtcnn.py`. Notebook dùng để học từng bước và có thể chạy trực tiếp nếu môi trường Jupyter cho phép mở cửa sổ OpenCV.

## 1. Chuẩn bị môi trường

**Input:** thư viện Python, thư mục ảnh tham chiếu `reference_faces/`.

**Output:** các biến cấu hình dùng xuyên suốt bài thực hành.

### 1.1. Cài đặt thư viện nếu thiếu

Chạy cell bên dưới khi gặp lỗi `ModuleNotFoundError`, ví dụ `No module named 'torch'`. Cell này dùng `sys.executable` để cài package đúng vào Python kernel đang chạy notebook. Nếu `pip` báo không tìm được bản `torch` phù hợp, hãy đổi kernel sang Python 3.10 hoặc 3.11 rồi chạy lại cell.

In [ ]:
# ============================================================
# PHAN 1.1: CAI DAT THU VIEN NEU THIEU
# Input : requirements.txt cua project hoac workaround cho Python moi
# Output: cac package duoc cai vao dung kernel Jupyter hien tai
# ============================================================

import subprocess
import sys
from pathlib import Path

requirements_path = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    candidate = folder / 'requirements.txt'
    if candidate.exists():
        requirements_path = candidate
        break

if requirements_path is None:
    raise FileNotFoundError('Kh?ng t?m th?y requirements.txt trong th? m?c hi?n t?i ho?c c?c th? m?c cha')

if sys.version_info >= (3, 13):
    print('[WARN] Kernel ?ang d?ng Python', sys.version.split()[0])
    print('[INFO] D?ng c?ch c?i t??ng th?ch Python m?i cho torch/facenet-pytorch.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch', 'torchvision', 'tqdm'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'facenet-pytorch', '--no-deps'])
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)])

In [ ]:
# ============================================================
# PHẦN 1.2: IMPORT THƯ VIỆN
# Input : các package đã cài bằng requirements.txt
# Output: cv2, np, torch, MTCNN, FaceNet sẵn sàng sử dụng
# ============================================================

from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization

print('[OK] Đã import thư viện')

In [ ]:
# ============================================================
# PHẦN 1.3: KHAI BÁO ĐƯỜNG DẪN VÀ THAM SỐ
# Input : cấu trúc thư mục của bài thực hành
# Output: REFERENCE_DIR, OUTPUT_DIR, THRESHOLD, DEVICE
# ============================================================

BASE_DIR = Path.cwd()
if BASE_DIR.name != 'PHAN_2':
    BASE_DIR = Path('CHUONG_3/PHAN_2') if Path('CHUONG_3/PHAN_2').exists() else BASE_DIR

REFERENCE_DIR = BASE_DIR / 'reference_faces'
OUTPUT_DIR = BASE_DIR / 'outputs'
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CAMERA_INDEX = 0
THRESHOLD = 0.7
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(f'[INFO] Thư mục ảnh tham chiếu: {REFERENCE_DIR.resolve()}')
print(f'[INFO] Thư mục lưu kết quả: {OUTPUT_DIR.resolve()}')
print(f'[INFO] Device: {DEVICE}')
print(f'[INFO] Ngưỡng similarity: {THRESHOLD}')

## 2. Truy cập webcam bằng OpenCV

**Input:** `CAMERA_INDEX`.

**Output:** một frame đọc được từ webcam. Nếu cell này lỗi, cần kiểm tra camera hoặc đổi `CAMERA_INDEX = 1`.

In [ ]:
# ============================================================
# PHẦN 2: KIỂM TRA WEBCAM
# Input : CAMERA_INDEX
# Output: frame đầu tiên từ webcam
# ============================================================

capture = cv2.VideoCapture(CAMERA_INDEX)
if not capture.isOpened():
    raise RuntimeError(f'Không mở được webcam index {CAMERA_INDEX}')

ok, frame_bgr = capture.read()
capture.release()

if not ok:
    raise RuntimeError('Webcam mở được nhưng không đọc được frame')

print('[OK] Đọc được frame từ webcam')
print('Kích thước frame:', frame_bgr.shape)

## 3. Tải MTCNN và FaceNet

**Input:** device CPU/GPU.

**Output:**

- `detector`: model MTCNN phát hiện khuôn mặt.
- `recognizer`: FaceNet/InceptionResnetV1 trích xuất embedding 512 chiều.

In [ ]:
# ============================================================
# PHẦN 3: KHỞI TẠO MODEL
# Input : DEVICE
# Output: detector, recognizer
# ============================================================

detector = MTCNN(keep_all=True, device=DEVICE)
recognizer = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)

print('[OK] Đã tải MTCNN và FaceNet/InceptionResnetV1')

## 4. Tạo embedding tham chiếu

Chép ảnh khuôn mặt cần nhận diện vào `reference_faces/`, hoặc chạy script:

```powershell
python CHUONG_3/PHAN_2/src/realtime_facenet_mtcnn.py --capture-reference --name sinh_vien
```

**Input:** các ảnh trong `reference_faces/`.

**Output:** `reference_embedding`, là vector trung bình của các ảnh tham chiếu hợp lệ.

In [ ]:
# ============================================================
# PHẦN 4.1: HÀM TIỀN XỬ LÝ, PHÁT HIỆN MẶT VÀ TRÍCH XUẤT EMBEDDING
# Input : ảnh RGB hoặc ảnh trong thư mục reference_faces
# Output: face crop, embedding chuẩn hóa L2
# ============================================================

def list_images(folder):
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ) if folder.exists() else []


def bgr_to_rgb(frame_bgr):
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


def clip_box(box, width, height, margin=20):
    x1, y1, x2, y2 = [int(round(value)) for value in box]
    x1 = max(0, x1 - margin)
    y1 = max(0, y1 - margin)
    x2 = min(width, x2 + margin)
    y2 = min(height, y2 + margin)
    return x1, y1, x2, y2


def crop_face(image_rgb, box, margin=20):
    height, width = image_rgb.shape[:2]
    x1, y1, x2, y2 = clip_box(box, width, height, margin)
    if x2 <= x1 or y2 <= y1:
        return None
    return image_rgb[y1:y2, x1:x2]


def face_to_tensor(face_rgb):
    face_resized = cv2.resize(face_rgb, (160, 160), interpolation=cv2.INTER_AREA)
    tensor = torch.from_numpy(face_resized).permute(2, 0, 1).float()
    tensor = fixed_image_standardization(tensor)
    return tensor.unsqueeze(0).to(DEVICE)


@torch.no_grad()
def get_embedding(face_rgb):
    embedding = recognizer(face_to_tensor(face_rgb))
    return F.normalize(embedding, p=2, dim=1)


def detect_largest_face(image_rgb):
    boxes, probabilities = detector.detect(image_rgb)
    if boxes is None or probabilities is None:
        return None
    valid_indices = [idx for idx, prob in enumerate(probabilities) if prob is not None and prob >= 0.90]
    if not valid_indices:
        return None
    return max(
        (boxes[idx] for idx in valid_indices),
        key=lambda box: (box[2] - box[0]) * (box[3] - box[1])
    )


def cosine_similarity(embedding_a, embedding_b):
    return float(F.cosine_similarity(embedding_a, embedding_b).item())

print('[OK] Đã khai báo hàm xử lý')

In [ ]:
# ============================================================
# PHẦN 4.2: ĐỌC ẢNH THAM CHIẾU VÀ TẠO EMBEDDING TRUNG BÌNH
# Input : REFERENCE_DIR
# Output: reference_embedding
# ============================================================

reference_paths = list_images(REFERENCE_DIR)
if not reference_paths:
    raise FileNotFoundError(
        f'Chưa có ảnh trong {REFERENCE_DIR.resolve()}. '
        'Hãy chạy: python CHUONG_3/PHAN_2/src/realtime_facenet_mtcnn.py --capture-reference --name sinh_vien '
        'hoặc chép ảnh vào thư mục này.'
    )

embeddings = []
for image_path in reference_paths:
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        print(f'[WARN] Không đọc được ảnh: {image_path.name}')
        continue

    image_rgb = bgr_to_rgb(image_bgr)
    box = detect_largest_face(image_rgb)
    if box is None:
        print(f'[WARN] Không phát hiện khuôn mặt rõ trong: {image_path.name}')
        continue

    face_rgb = crop_face(image_rgb, box)
    embeddings.append(get_embedding(face_rgb))
    print(f'[OK] Đã lấy embedding: {image_path.name}')

if not embeddings:
    raise RuntimeError('Không tạo được embedding tham chiếu từ ảnh hiện có')

reference_embedding = torch.mean(torch.cat(embeddings, dim=0), dim=0, keepdim=True)
reference_embedding = F.normalize(reference_embedding, p=2, dim=1)

print('[OUTPUT] Kích thước reference_embedding:', tuple(reference_embedding.shape))

## 5. Nhận diện khuôn mặt thời gian thực

**Input:** webcam frame, MTCNN, FaceNet, `reference_embedding`.

**Output:** cửa sổ webcam có bounding box và nhãn:

- `Matched`: khi `similarity > 0.7`.
- `Unknown`: khi `similarity <= 0.7`.

Nhấn `p` để lưu frame kết quả, nhấn `q` để thoát.

In [ ]:
# ============================================================
# PHẦN 5: VÒNG LẶP NHẬN DIỆN THỜI GIAN THỰC
# Input : webcam + reference_embedding
# Output: cửa sổ hiển thị Matched/Unknown theo similarity
# ============================================================

def draw_result(frame_bgr, box, label, similarity, color):
    height, width = frame_bgr.shape[:2]
    x1, y1, x2, y2 = clip_box(box, width, height, margin=0)
    cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color, 2)
    cv2.putText(
        frame_bgr,
        f'{label} {similarity:.2f}',
        (x1, max(25, y1 - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.75,
        color,
        2,
    )


capture = cv2.VideoCapture(CAMERA_INDEX)
if not capture.isOpened():
    raise RuntimeError(f'Không mở được webcam index {CAMERA_INDEX}')

print("[INFO] Nhấn 'p' để lưu frame, 'q' để thoát")

try:
    while True:
        ok, frame_bgr = capture.read()
        if not ok:
            print('[WARN] Không đọc được frame từ webcam')
            break

        frame_rgb = bgr_to_rgb(frame_bgr)
        boxes, probabilities = detector.detect(frame_rgb)

        if boxes is not None and probabilities is not None:
            for box, probability in zip(boxes, probabilities):
                if probability is None or probability < 0.90:
                    continue

                face_rgb = crop_face(frame_rgb, box)
                if face_rgb is None:
                    continue

                embedding = get_embedding(face_rgb)
                similarity = cosine_similarity(embedding, reference_embedding)

                # Điều kiện so sánh đúng yêu cầu đề bài.
                if similarity > THRESHOLD:
                    label = 'Matched'
                    color = (0, 220, 0)
                else:
                    label = 'Unknown'
                    color = (0, 0, 255)

                draw_result(frame_bgr, box, label, similarity, color)

        cv2.putText(
            frame_bgr,
            f'Threshold: {THRESHOLD:.2f}',
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (255, 255, 0),
            2,
        )
        cv2.imshow('FaceNet + MTCNN realtime recognition', frame_bgr)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('p'):
            output_path = OUTPUT_DIR / f"recognition_{datetime.now().strftime('%Y%m%d_%H%M%S')}.jpg"
            cv2.imwrite(str(output_path), frame_bgr)
            print(f'[OK] Đã lưu: {output_path}')
        elif key == ord('q'):
            break
finally:
    capture.release()
    cv2.destroyAllWindows()

## 6. Nhận xét và mở rộng

**Kết quả cần đạt:** webcam hiển thị khung khuôn mặt, điểm similarity và nhãn `Matched` hoặc `Unknown`.

**Bài tập mở rộng:**

1. Thử các ngưỡng `0.6`, `0.7`, `0.8` và nhận xét số lần nhận đúng/sai.
2. Thêm nhiều ảnh tham chiếu ở các góc mặt khác nhau.
3. Lưu log similarity theo thời gian để vẽ biểu đồ.
4. Mở rộng từ 1 người tham chiếu sang nhiều người bằng cách tạo embedding riêng cho từng thư mục.